# Week 3 + Week 4 — Serving: Redis feature store, ANN search, FastAPI, stress test

**Run this after Week 1 and Week 2** (it needs `hm_project/week2_model/` in Google Drive).

| Step | Week | What happens |
|---|---|---|
| 1 | — | Mount Drive, get the project code, install Redis + libraries |
| 2 | Week 3 Day 7 | Build the FAISS HNSW (ANN) index over the item embeddings |
| 3 | Week 3 Day 4-6 | Load user features + item vectors into Redis (feature store) |
| 4 | Week 4 Day 1-3 | Start the FastAPI service and call it |
| 5 | Week 4 Day 6-7 | Automated API tests + stress test under concurrent load |
| 6 | Week 4 Day 4-5 | Airflow DAGs (weekly retraining, daily embedding refresh) |

Runtime → Run all. A CPU runtime is enough.

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# Get the project code (branch with all 4 weeks)
!rm -rf /content/hm-recommendation-engine
!git clone -q -b final-project https://github.com/afnakurikkal/hm-recommendation-engine.git /content/hm-recommendation-engine
%cd /content/hm-recommendation-engine
!ls

In [ ]:
!apt-get -qq update > /dev/null && apt-get -qq install -y redis-server > /dev/null
!pip install -q faiss-cpu fastapi uvicorn redis pytest
!redis-server --daemonize yes
!sleep 1 && redis-cli ping

In [ ]:
import os
MODEL_DIR = "/content/drive/MyDrive/hm_project/week2_model"
needed = ["query_tower.weights.h5", "candidate_tower.weights.h5", "item_embeddings.npy",
          "item_ids.json", "trained_user_idx.npy", "model_config.json"]
missing = [f for f in needed if not os.path.exists(os.path.join(MODEL_DIR, f))]
print("Missing Week 2 files:", missing or "none - ready!")
assert not missing, "Run the Week 2 notebook first."

## 2. Week 3 — ANN index (FAISS HNSW)

In [ ]:
!python -m src.serving.ann_search

## 3. Week 3 — Redis feature store
Loads every customer's model features and every article's vector + metadata into Redis.
For a fast demo it loads 200,000 customers (customers with purchases first). Delete the `HM_MAX_USERS` line to load all.

In [ ]:
%env HM_MAX_USERS=200000
# ^ loads 200k customers for a fast demo; delete that line to load all 1.37M
!python -m src.serving.load_redis

## 4. Week 4 — FastAPI recommendation service

In [ ]:
!pkill -f "uvicorn src.api.main" || true
!nohup uvicorn src.api.main:app --host 0.0.0.0 --port 8000 > api.log 2>&1 &

import time, requests
for _ in range(90):
    try:
        if requests.get("http://127.0.0.1:8000/health", timeout=2).status_code == 200:
            break
    except requests.RequestException:
        time.sleep(2)
print(requests.get("http://127.0.0.1:8000/health").json())

### Frontend + backend links
Click the **frontend** link to open the web app (enter a customer ID or use the "Try" buttons).
The **backend** link opens the interactive API docs. Both work while this notebook is running.

In [ ]:
from google.colab.output import eval_js, serve_kernel_port_as_window
base = eval_js("google.colab.kernel.proxyPort(8000)")
print("FRONTEND (web app):   ", base + "ui")
print("BACKEND  (API docs):  ", base + "docs")
serve_kernel_port_as_window(8000, path="/ui", anchor_text="Open the H&M recommendation web app")

In [ ]:
import json, redis, pandas as pd
r = redis.Redis()
API = "http://127.0.0.1:8000"

def find_customer(new_customer):
    for key in r.scan_iter("user:*", count=1000):
        if (r.hget(key, "is_new_user") == b"1") == new_customer:
            return key.decode().split(":", 1)[1]

def show(customer_id, title):
    data = requests.get(f"{API}/recommend/{customer_id}", params={"k": 12}).json()
    print(f"\n=== {title}\ncustomer: {customer_id[:20]}... | source: {data['source']} | "
          f"profile: {data['customer']} | latency: {data['latency_ms']} ms")
    display(pd.DataFrame(data["recommendations"]))

show(find_customer(False), "Existing customer (two-tower model)")
show(find_customer(True) or "none", "New customer, no purchases (model uses profile features)")
show("unknown-customer-123", "Unknown customer (popular-items fallback)")

In [ ]:
# Item-to-item: articles similar to one article (ANN search on item vectors)
article_id = json.load(open(f"{MODEL_DIR}/item_ids.json"))[1000]
data = requests.get(f"{API}/similar/{article_id}", params={"k": 6}).json()
print("query article:", article_id, r.hget(f"item:{article_id}", "product_type_name").decode(),
      "/", r.hget(f"item:{article_id}", "colour_group_name").decode(), f"| {data['latency_ms']} ms")
display(pd.DataFrame(data["similar"]))

## 5. Week 4 — Automated API tests and stress test

In [ ]:
!python -m pytest tests/test_api.py -v

In [ ]:
!python -m tests.stress_test

## 6. Week 4 — Airflow pipelines
`dags/hm_recommendation_dags.py` defines two DAGs:

- **hm_weekly_retraining** (Sunday 02:00): build_features → train_two_tower → build_ann_index → load_feature_store → reload_api → api_smoke_test
- **hm_daily_embedding_refresh** (daily 03:00): build_features → refresh_item_embeddings → build_ann_index → load_feature_store → reload_api

Each task runs one module from `src/`, the same code used in this notebook. The cell below shows the DAG code.
The next one is optional: it installs Airflow (a few minutes) and checks that both DAGs load.

In [ ]:
!cat dags/hm_recommendation_dags.py

In [ ]:
# OPTIONAL: install Airflow and check the DAGs load (skip if short on time)
!pip install -q apache-airflow
import os
os.environ["AIRFLOW__CORE__DAGS_FOLDER"] = "/content/hm-recommendation-engine/dags"
os.environ["AIRFLOW__CORE__LOAD_EXAMPLES"] = "False"
!airflow db migrate > /dev/null 2>&1
!airflow dags list 2>/dev/null | grep hm_
!airflow tasks list hm_weekly_retraining 2>/dev/null

In [ ]:
# API log (useful if something failed)
!tail -n 20 api.log